# Bonus — Generate Synthetic Document-VLM Deployment Logs

**Hands-on synthetic data pipeline workshop with NeMo Data Designer and NeMo Anonymizer — PyData London 2026**

This bonus notebook synthesises multi-turn deployment logs for a hypothetical
**rich-business-document VLM** — the kind of model whose training data
Notebook 2 produces from `data/rich_document_seed.parquet`. Each row contains
three independent PII surfaces — an application system prompt, an attached
business document (clinical trial report, financial variance memo, market
research brief, product launch readiness plan, operations dashboard export,
or customer support incident review), and a full multi-turn ChatML
conversation trace with tool calls and tool-result JSON — so Notebook 3 has
a realistic anonymization target.

## What it produces

A parquet file at `data/usage_logs_seed.parquet` containing `system_prompt`,
`attached_document_context`, and `conversation_trace` columns plus the
persona / intent / tone samplers that drove generation. Notebook 3 reads
this file via `load_usage_logs_seed()`.

## What you'll learn

- **Chained `LLMTextColumnConfig` columns** — three text columns, each Jinja-templated against upstream columns and resolved as a DAG by Data Designer
- **In-DAG `CustomColumnConfig` post-processing** — pair each raw LLM column with a `@custom_column_generator` Python function that parses, sanitizes, and re-serializes the output, all inside the Data Designer DAG (no off-pipeline cleanup step)
- **Multi-turn ChatML in one shot** — synthesise a complete conversation trace (user turns, assistant tool calls in OpenAI/Nemotron `type:"custom"` format, tool result messages) as a single JSON string per row
- **Disabling reasoning mode** — `nemotron-3-super` with `enable_thinking=False` to avoid `<think>`-block leakage in `content`
- **Output validation** — custom column returns `None` for un-parseable rows; the export cell drops NaN with one `dropna` call

> **Prerequisites**: this notebook is optional; the workshop ships with `data/usage_logs_seed.parquet` already checked in. Allow ~10–14 minutes end-to-end against NVIDIA Build's free tier.

## Setup

In [ ]:
from pathlib import Path

from notebook_helpers import (
    DATA_DIR,
    build_dd_model_setup,
    environment_setup,
    show_provider_info,
)

provider = environment_setup(provider="auto")
show_provider_info(provider)

import data_designer.config as dd
from data_designer.interface import DataDesigner

model_providers, model_configs = build_dd_model_setup(provider)
config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)
data_designer = DataDesigner(model_providers=model_providers)

## Prompts

Three prompt strings drive the chained LLM columns. Edit them here to tune the output.

In [ ]:
LOG_SYSTEM_PROMPT_PROMPT = """\
You produce the raw application-built **system prompt** that a SaaS
business-document analytics product injects ahead of every chat turn with
its LLM/VLM backend. In real products this string carries the user's
identity, account context, and personalisation hooks, and it is logged
verbatim alongside every request.

**User persona:** {{ user_persona }}
**Task intent likely for this session:** {{ task_intent }}
**Tone the user tends to write in:** {{ urgency_tone }}

**Your task:** write the system prompt as the backend would interpolate
it at request time. Include all of these:

1. A first paragraph naming the product (invent a plausible name like
   "DocPulse Analytics", "BriefMatter", "BoardLens AI", "KPI Copilot",
   "ReviewLens") and its capability scope (summarise rich business
   documents, extract KPIs from charts/tables, answer questions about
   uploaded reports).
2. A second paragraph identifying the authenticated user with **all** of:
   - full name
   - work email address
   - employer / company name
   - their role / department / city
   - the project, ticket, or quarter they are currently working on (an
     internal codename like "Project Atlas", "Q3 close review", or
     "Roadmap-2026-Wave-2" is realistic)
   - 1-2 colleagues by name + role (e.g. "their manager Sarah Chen, VP
     Operations, and analytics partner Rohan Mehta")
3. A third short paragraph with style guidance: address the user by first
   name, ground every claim in the uploaded document, cite chart labels or
   table cells when answering numeric questions, etc.

**Hard rules:**
- Use plausible synthetic identifiers. Vary geography across rows
  (US / UK / EU / Canada / India / Australia). Do NOT use real public
  companies or real public people.
- 90-180 words. Plain text. No markdown headings. No "Output:" preambles.
- Do NOT redact or mask any identifiers; production system prompts
  preserve them verbatim.
- Output ONLY the final system prompt text. Do NOT include any reasoning,
  planning, "let's think", word counts, or meta-commentary.

**Output:** Just the system prompt body, ready to send.
""".strip()


LOG_DOCUMENT_CONTEXT_PROMPT = """\
You produce a short plain-text rendering of a rich business document a
user has uploaded to a deployed business-document VLM. Treat it as the
text the VLM would "see" after a quick OCR / layout-extraction pass over
the uploaded page.

**Persona context:** the user is a {{ user_persona }} whose current task
is {{ task_intent }}. The document type for this row is **{{ document_type }}**.

Use this cheat sheet for what each document type contains:

- clinical trial status report -- principal investigator, site IDs,
  enrollment by arm, adverse-event counts, cohort dates
- financial variance memo -- department, owner, prior-period vs current
  figures with currency, variance percentage, commentary
- market research brief -- segment, audience, win-loss themes, sample
  size, fielding window
- product launch readiness plan -- workstream, owner, gate status,
  go/no-go criteria, target launch date
- operations dashboard export -- region, KPI name, current value, target,
  alert threshold, last-updated timestamp
- customer support incident review -- incident ID, reporter, owner,
  severity, opened/closed timestamps, root cause, remediation owner

**Your task:** produce a brief, faithful rendering of one document of the
specified type, populated with realistic-looking but fully synthetic
identifiers. Include all of these:
- A clear document title naming the document type and a period or codename
- Organization (employer / vendor / partner) name and at least one of
  address / email / phone / web domain
- 2-4 named stakeholders (full name + role + email) -- e.g. document
  owner, primary contact, approver, reviewer
- At least one document identifier (`Trial-A2-2026-Q1`, `Memo-FIN-1429`,
  `MRB-EMEA-LowEnt-44`, `Launch-Plan-Phoenix`, `OPS-EMEA-2026-W19`,
  `INC-2026-0822`, etc.)
- 3-6 numeric or factual data points relevant to the document type
  (enrollment counts, dollar variances, win rates, gate statuses, KPI
  values, severity levels)
- Dates / timestamps where appropriate

**Hard rules:**
- Use realistic-looking names, addresses, and identifiers -- but do NOT
  use the names of real public companies or real public people.
- Vary geography across rows: rotate through US, UK, EU (France /
  Germany / Spain / Italy / Netherlands), Canada, India, Australia.
- Keep it under 220 words. Plain text. No markdown headings, no asterisks.
- Do NOT redact, mask, or anonymise anything; uploaded documents preserve
  their original identifiers verbatim.
- Output ONLY the final document text. Do NOT include any reasoning,
  planning, "let's think", word counts, or meta-commentary. No <think>
  blocks.

**Output:** Just the document text. No preamble, no closing remarks.
""".strip()


LOG_CONVERSATION_TRACE_PROMPT = """\
**OUTPUT FORMAT (READ FIRST):** Your response must begin with the character `{`
and end with the character `}`. Output nothing else -- no markdown code
fences, no `<think>` blocks, no "Let me work this out", no word counts, no
prose before or after the JSON. If you find yourself wanting to plan, do it
silently. Just emit the JSON.

You are simulating a **realistic multi-turn deployment log** of one chat
session between a user and the deployed business-document VLM. Output the
full conversation trace as a single JSON document in OpenAI-messages format.

**Session context:**
- user_persona:  {{ user_persona }}
- task_intent:   {{ task_intent }}
- document_type: {{ document_type }}
- tone:          {{ urgency_tone }}

**The application's system prompt (must appear verbatim as `messages[0]`):**
{{ system_prompt }}

**The document the user uploaded (already attached; the model may quote it):**
{{ attached_document_context }}

**Your task:** Produce a JSON object with a single key `messages` whose value
is an array of 6-10 message objects representing the **full** conversation
trace as a real production log would store it.

**The first message MUST be the system message verbatim from the system
prompt above** -- this matches how OpenAI/Nemotron API logs look on disk:
every request carries the system message at index 0. Use
``{"role": "system", "content": "<verbatim system prompt>"}``.

The remaining 5-9 messages are the user/assistant/tool turns described
below.

**Realism requirements -- read these carefully:**

1. **The user is already authenticated.** The system prompt above already
   names them, their email, their employer. Real users do NOT re-introduce
   themselves to a chat assistant they are signed into. So user messages
   should be **terse and task-focused** -- 1-3 sentences each, mostly
   referencing identifiers from the *uploaded document* (organization
   name, document ID, named stakeholders, key metrics) rather than
   identifying themselves. Roughly 1 in 4 user turns may casually mention
   a colleague by first name ("can you forward this to Sarah?") or an
   internal project codename.

2. **Multi-turn = the user iterates.** Include 2-3 user turns total: an
   initial ask, then a follow-up that probes deeper, refines the request,
   or pushes back on something the model said. Real sessions are not
   one-shot.

3. **The assistant uses tools.** Include 1-3 assistant tool calls across
   the conversation. Pick from this catalogue (use the exact tool names):
   - `lookup_organization_record(organization_name)` -- returns CRM
     record with industry, headcount, ARR band, primary contact
   - `lookup_owner_contact(role, organization)` -- returns named owner
     with email and direct phone
   - `compare_to_baseline(metric, period)` -- returns prior-period value
     and delta with commentary
   - `validate_compliance(document_id, framework)` -- returns compliance
     status against named framework (SOC2, GDPR, HIPAA, ISO27001, etc.)
   - `flag_for_review(document_id, reason)` -- returns review ticket ID
     with assigned reviewer
   - `notify_stakeholder(email, message)` -- returns notification log
     entry with timestamp

4. **Tool calls and tool results carry their own identifiers.** Tool
   argument JSON will quote organization names, document IDs, recipient
   emails. Tool results should return plausible-looking JSON containing
   additional identifiers the user never typed -- internal CRM contacts,
   approver emails, reviewer names, contract IDs, ARR bands. Real
   production logs preserve these results verbatim.

5. **The assistant addresses the user by first name** (per the system
   prompt's instructions) and grounds every claim in the document + tool
   results.

**Required JSON schema:**

```
{
  "messages": [
    {"role": "system", "content": "<verbatim system prompt from above>"},
    {"role": "user", "content": "<terse user message>"},
    {"role": "assistant", "content": null,
     "tool_calls": [
       {"id": "call_<short hex>", "type": "custom",
        "custom": {"name": "<tool name>", "input": "<JSON-stringified args>"}}
     ]},
    {"role": "tool", "tool_call_id": "call_<same id>",
     "content": "<JSON-stringified tool result>"},
    {"role": "assistant", "content": "<reply that uses the tool result>"},
    {"role": "user", "content": "<follow-up>"},
    {"role": "assistant", "content": "<final reply>"}
  ]
}
```

`tool_calls` and `tool_call_id` connect via the matching `id`. The `input`
field on each tool call and the `content` of each tool message are
JSON-stringified inner JSON (so they're strings containing JSON, not nested
JSON objects).

**Hard rules:**
- Output **only** the JSON document. No prose, no code fences, no <think>
  blocks, no commentary before or after.
- Do NOT redact, mask, or anonymise anything inside the trace; production
  logs preserve identifiers verbatim.
- Do NOT use the names of real public companies or real public people.
- Keep total trace length under ~600 words so it fits comfortably in a
  parquet cell.

**Output:** A single valid JSON object. Nothing else.
""".strip()


## Build configuration: Multi-turn deployment logs

Four sampler columns drive who/what/how/which-doctype, then three chained LLM-text columns produce the three PII surfaces. Each LLM column is paired with a `CustomColumnConfig` sanitizer that runs in-DAG: the raw LLM output (`_system_prompt_raw`, `_attached_document_context_raw`, `_conversation_trace_raw`) is dropped from the final dataset, and the published columns (`system_prompt`, `attached_document_context`, `conversation_trace`) are the parsed-and-sanitized versions. The trace LLM column references the *sanitized* system prompt and document context in its Jinja template, so `messages[0]` lands quote-clean. Data Designer resolves the dependency DAG automatically.

In [ ]:
import json


# ---- JSON helpers used by the trace sanitizer below -------------------------
def _balanced_json_object(text: str) -> str | None:
    """Return the first balanced top-level {...} JSON object in `text`, or None."""
    start = text.find("{")
    if start < 0:
        return None
    depth = 0
    in_str = False
    escape = False
    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return text[start : i + 1]
    return None


def parse_trace(raw):
    """Parse an LLM-generated trace string. Returns the dict or None on failure."""
    if isinstance(raw, dict):
        return raw if "messages" in raw else None
    if not isinstance(raw, str) or not raw.strip():
        return None
    text = raw.strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text[:5].lower().startswith("json"):
            text = text[4:].lstrip()
    candidate = _balanced_json_object(text) or text
    try:
        parsed = json.loads(candidate)
    except json.JSONDecodeError:
        return None
    if not isinstance(parsed, dict) or "messages" not in parsed:
        return None
    return parsed


def _sanitize_str(s):
    """Replace embedded literal `\"` with `'` inside string values."""
    return s.replace('"', "'") if isinstance(s, str) else s


def _sanitize_obj(obj):
    if isinstance(obj, dict):
        return {k: _sanitize_obj(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_sanitize_obj(v) for v in obj]
    return _sanitize_str(obj)


# ---- Sampler columns: who/what/how ------------------------------------------
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="user_persona",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "product manager",
                "operations analyst",
                "compliance officer",
                "finance analyst",
                "people analytics lead",
                "engineering manager",
            ],
            weights=[0.22, 0.20, 0.15, 0.18, 0.13, 0.12],
        ),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="task_intent",
        sampler_type=dd.SamplerType.SUBCATEGORY,
        params=dd.SubcategorySamplerParams(
            category="user_persona",
            values={
                "product manager":         ["summarize_quarterly_review", "extract_kpi_targets", "compare_to_roadmap"],
                "operations analyst":      ["identify_outliers", "extract_trend_signal", "summarize_incidents"],
                "compliance officer":      ["extract_audit_evidence", "validate_regulatory_alignment", "flag_anomaly"],
                "finance analyst":         ["extract_variance", "compare_to_budget", "validate_forecast"],
                "people analytics lead":   ["extract_funnel_metrics", "summarize_attrition", "identify_pipeline_gaps"],
                "engineering manager":     ["summarize_incident", "extract_remediation_owner", "compare_to_sla"],
            },
        ),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="urgency_tone",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["matter-of-fact", "rushed", "frustrated", "thorough"],
            weights=[0.45, 0.25, 0.10, 0.20],
        ),
        drop=True,  # used only as a Jinja input to LLM prompts; not in final parquet
    )
)

# Document type is conditional on persona, so we model it the same way
# task_intent is modelled: a SUBCATEGORY sampler keyed off user_persona.
# Guarantees a finance analyst gets a finance-shaped doc and a compliance
# officer gets compliance-shaped material -- no LLM "pick one" step needed.
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="document_type",
        sampler_type=dd.SamplerType.SUBCATEGORY,
        params=dd.SubcategorySamplerParams(
            category="user_persona",
            values={
                "product manager":       ["market research brief", "product launch readiness plan", "operations dashboard export"],
                "operations analyst":    ["operations dashboard export", "customer support incident review", "financial variance memo"],
                "compliance officer":    ["clinical trial status report", "customer support incident review", "financial variance memo"],
                "finance analyst":       ["financial variance memo", "operations dashboard export", "market research brief"],
                "people analytics lead": ["market research brief", "operations dashboard export", "customer support incident review"],
                "engineering manager":   ["customer support incident review", "product launch readiness plan", "operations dashboard export"],
            },
        ),
    )
)

# ---- Raw LLM text columns (dropped from final dataset) -----------------------
# These three columns hold the raw LLM output. Each is paired with a custom
# column downstream that sanitizes it (recursive `\"` -> `'` swap inside string
# values) into the publishable, identically-named clean column.
config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="_system_prompt_raw",
        prompt=LOG_SYSTEM_PROMPT_PROMPT,
        model_alias=provider.text_alias,
        drop=True,
    )
)

config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="_attached_document_context_raw",
        prompt=LOG_DOCUMENT_CONTEXT_PROMPT,
        model_alias=provider.text_alias,
        drop=True,
    )
)


# ---- Custom-column sanitizers for the two text surfaces ---------------------
@dd.custom_column_generator(required_columns=["_system_prompt_raw"])
def sanitize_system_prompt(row):
    return _sanitize_str(row["_system_prompt_raw"])


@dd.custom_column_generator(required_columns=["_attached_document_context_raw"])
def sanitize_attached_document_context(row):
    return _sanitize_str(row["_attached_document_context_raw"])


config_builder.add_column(
    dd.CustomColumnConfig(
        name="system_prompt",
        generator_function=sanitize_system_prompt,
    )
)

config_builder.add_column(
    dd.CustomColumnConfig(
        name="attached_document_context",
        generator_function=sanitize_attached_document_context,
    )
)


# ---- Trace LLM column references the SANITIZED system_prompt + doc context --
# This way the verbatim system message the model copies into messages[0] is
# already quote-clean, matching the rest of the trace.
config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="_conversation_trace_raw",
        prompt=LOG_CONVERSATION_TRACE_PROMPT,
        model_alias=provider.text_alias,
        drop=True,
    )
)


# ---- Final trace sanitizer: parse -> recursive `\"` -> `'` -> re-serialize --
# Returns None when parse_trace fails, surfacing as NaN in the final
# DataFrame so the export cell can drop it with a single dropna().
@dd.custom_column_generator(required_columns=["_conversation_trace_raw"])
def sanitize_conversation_trace(row):
    parsed = parse_trace(row["_conversation_trace_raw"])
    if parsed is None:
        return None
    return json.dumps(_sanitize_obj(parsed), ensure_ascii=False)


config_builder.add_column(
    dd.CustomColumnConfig(
        name="conversation_trace",
        generator_function=sanitize_conversation_trace,
    )
)

data_designer.validate(config_builder)


### Preview

In [ ]:
preview = data_designer.preview(config_builder, num_records=2)
preview.display_sample_record()

In [ ]:
# Re-run this cell (Shift+Enter) to cycle through preview rows.
preview.display_sample_record()

In [ ]:
import json
import pandas as pd

# `conversation_trace` is now the sanitized custom column: either a valid
# JSON string, or NaN if the upstream raw LLM output failed to parse.
for i, raw in enumerate(preview.dataset["conversation_trace"].tolist()):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        print(f"Row {i}: \u2717 raw LLM JSON failed to parse (custom-column sanitizer returned NaN)")
        continue
    msgs = json.loads(raw)["messages"]
    n_tool_calls = sum(1 for m in msgs if m.get("tool_calls"))
    n_tool_results = sum(1 for m in msgs if m.get("role") == "tool")
    print(f"Row {i}: \u2713 {len(msgs)} messages, {n_tool_calls} tool call(s), {n_tool_results} tool result(s)")

## Create full usage-logs dataset

In [ ]:
results = data_designer.create(
    config_builder,
    num_records=25,
    dataset_name="document_vlm_usage_logs",
)
df = results.load_dataset()
print(f"Generated {len(df)} rows.")
df.head(3)

## Export usage logs seed parquet

Drop rows where the model produced invalid JSON, then write the columns Notebook 3 reads via `load_usage_logs_seed()`.

In [ ]:
KEEP_COLUMNS = [
    "user_persona",
    "task_intent",
    "document_type",
    "system_prompt",
    "attached_document_context",
    "conversation_trace",
]

# `conversation_trace` is the parsed-and-sanitized output of the custom column
# downstream of `_conversation_trace_raw`. Rows where the LLM produced
# un-parseable JSON surface here as NaN, so the validity filter is a one-liner.
TARGET_ROWS = 20
clean = df.dropna(subset=["conversation_trace"]).reset_index(drop=True)[KEEP_COLUMNS]
n_dropped = len(df) - len(clean)
print(f"{len(clean)} valid / {len(df)} total ({n_dropped} dropped due to malformed JSON).")

if len(clean) < TARGET_ROWS:
    raise RuntimeError(
        f"Only {len(clean)} valid rows available; need {TARGET_ROWS}. "
        f"Increase num_records in the create cell and re-run."
    )
clean = clean.head(TARGET_ROWS).reset_index(drop=True)

DATA_DIR.mkdir(parents=True, exist_ok=True)
out_path = DATA_DIR / "usage_logs_seed.parquet"
clean.to_parquet(out_path, index=False)
print(f"Wrote {len(clean)} rows (TARGET_ROWS={TARGET_ROWS}) to {out_path.relative_to(Path.cwd().parent)}.")
